In [1]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="experiment_1b",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

In [2]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

MODEL = "gpt-4.1-mini"

LABEL_MAP = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}
ALLOWED_LABELS = list(LABEL_MAP.values())


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
data = vault.query_item_content("ag_news_small")
df = pd.DataFrame(data)

df.head()


---[ TableVault Record ]---


,text,label
0,Indian board plans own telecast of Australia s...,1
1,Stocks Higher on Drop in Jobless Claims A shar...,2
2,"Nuggets 112, Raptors 106 Carmelo Anthony score...",1
3,Stocks Higher on Drop in Jobless Claims A shar...,2
4,REVIEW: 'Half-Life 2' a Tech Masterpiece (AP) ...,3


---[ TableVault Record ]---



In [5]:
def extract_first_json_object(raw_text: str) -> dict:
    if not raw_text or not raw_text.strip():
        raise ValueError("Model returned empty output.")

    decoder = json.JSONDecoder()
    for i, ch in enumerate(raw_text):
        if ch != "{":
            continue
        try:
            candidate, _ = decoder.raw_decode(raw_text[i:])
        except json.JSONDecodeError:
            continue
        if isinstance(candidate, dict):
            return candidate

    raise ValueError(f"Could not find JSON object in model output: {raw_text!r}")


def extract_reasoning(text: str) -> dict:
    prompt = f"""
You are analyzing a news article before classification.

Possible final labels later will be:
- World
- Sports
- Business
- Sci/Tech

Read the article and return strict JSON with keys:
- summary
- main_topic
- evidence
- likely_label_candidates

Rules:
- summary: 1 short sentence
- main_topic: short phrase
- evidence: short explanation of what in the text matters
- likely_label_candidates: a JSON list of 1 to 3 labels chosen only from
  ["World", "Sports", "Business", "Sci/Tech"]

Do not use markdown code fences.

Article:
{text}
""".strip()

    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    raw = response.output_text or ""
    print(raw)
    result = extract_first_json_object(raw)

    if not isinstance(result.get("summary"), str):
        raise ValueError("Missing or invalid 'summary' in model output.")

    if not isinstance(result.get("main_topic"), str):
        raise ValueError("Missing or invalid 'main_topic' in model output.")

    if not isinstance(result.get("evidence"), str):
        raise ValueError("Missing or invalid 'evidence' in model output.")

    candidates = result.get("likely_label_candidates")
    if not isinstance(candidates, list):
        raise ValueError("Missing or invalid 'likely_label_candidates' in model output.")

    for label in candidates:
        if label not in ALLOWED_LABELS:
            raise ValueError(f"Unexpected candidate label: {label}")

    return {
        "summary": result["summary"],
        "main_topic": result["main_topic"],
        "evidence": result["evidence"],
        "likely_label_candidates": candidates,
    }


---[ TableVault Record ]---
---[ TableVault Record ]---



In [6]:
def classify_from_reasoning(reasoning: dict) -> dict:
    prompt = f"""
You are a careful news classifier.

Choose exactly one final label from:
- World
- Sports
- Business
- Sci/Tech

You are given a prior reasoning object.

Return strict JSON with keys:
- predicted_label
- short_reason

Do not use markdown code fences.

Reasoning object:
{json.dumps(reasoning, ensure_ascii=False, indent=2)}
""".strip()

    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    raw = response.output_text or ""
    print(raw)
    result = extract_first_json_object(raw)

    if result.get("predicted_label") not in ALLOWED_LABELS:
        raise ValueError(f"Unexpected label: {result.get('predicted_label')}")

    if not isinstance(result.get("short_reason"), str):
        raise ValueError("Missing or invalid 'short_reason' in model output.")

    return {
        "predicted_label": result["predicted_label"],
        "short_reason": result["short_reason"],
    }


---[ TableVault Record ]---
---[ TableVault Record ]---



In [7]:
vault.create_record_list("reason_output", column_names=["gold_label", 
                                                        "summary", 
                                                        "main_topic", 
                                                        "evidence",
                                                        "label_candidates",
                                                        "predicted_label",
                                                        "short_reason"])

description = (
    "Per-article reasoning and classification outputs for experiment_1b on the ag_news_small list. "
    "Each record comes from one source news item and stores the gold label, model-generated summary, "
    "main topic, supporting evidence, likely label candidates, final predicted label, and short rationale. "
    "Use this list to inspect reasoning traces, compare predicted versus gold categories, analyze errors, "
    "search by topic or evidence language, and trace each record back to its originating ag_news_small row."
)
embedding = get_embeddings(description)
vault.create_description("reason_output", description, embedding)

for i, row in df.iterrows():
    text = row["text"][:1000]

    reasoning = extract_reasoning(text)
    final_pred = classify_from_reasoning(reasoning)
    input_items = {"ag_news_small": [i, i + 1]}
    vault.append_record("reason_output",
                        {
                            "gold_label": LABEL_MAP[row["label"]],
                            "summary": reasoning["summary"],
                            "main_topic": reasoning["main_topic"],
                            "evidence": reasoning["evidence"],
                            "label_candidates": reasoning["likely_label_candidates"],
                            "predicted_label": final_pred["predicted_label"],
                            "short_reason": final_pred["short_reason"],
                        },
                        input_items = input_items
                       )



---[ TableVault Record ]---
{
  "summary": "The Indian cricket board is arranging its own broadcast for the upcoming Australia test series amid a TV rights dispute.",
  "main_topic": "Cricket broadcasting dispute",
  "evidence": "The Indian cricket board is making arrangements to broadcast the Australia test series independently due to a TV rights dispute.",
  "likely_label_candidates": ["Sports", "Business"]
}
{
  "predicted_label": "Sports",
  "short_reason": "The main topic revolves around cricket broadcasting, which is directly related to a sport event."
}
{
  "summary": "Stocks rose due to a drop in jobless claims and positive forecasts from tech companies.",
  "main_topic": "Stock market response to economic and corporate news",
  "evidence": "The article highlights a sharp drop in initial unemployment claims and bullish forecasts from Nokia and Texas Instruments influencing stock price movement.",
  "likely_label_candidates": ["Business", "Sci/Tech"]
}
{
  "predicted_label": "B

In [8]:
results_df = pd.DataFrame(vault.query_item_content("reason_output"))
results_df["correct"] = results_df["gold_label"] == results_df["predicted_label"]
accuracy = results_df["correct"].mean()
vault.create_record_list("experiment_1b_accuracy", column_names=["accuracy"])
description = (
    "Experiment-level evaluation summary for experiment_1b. This list stores the aggregate classification "
    "accuracy computed from reason_output by comparing gold_label and predicted_label across the processed "
    "ag_news_small articles. The stored value is useful for retrieving overall model performance, benchmark "
    "comparisons, and experiment summaries rather than per-article reasoning details."
)
embedding = get_embeddings(description)
vault.create_description("experiment_1b_accuracy", description, embedding)
input_items = {"reason_output": [0, len(results_df)]}
vault.append_record("experiment_1b_accuracy", {"accuracy": accuracy}, input_items=input_items)
accuracy


---[ TableVault Record ]---


np.float64(0.9)

---[ TableVault Record ]---

